In [1]:
import json
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import time
from rdkit import Chem
from rdkit import RDLogger
from torch_geometric.nn import GCNConv
from scipy.stats import pearsonr

RDLogger.DisableLog('rdApp.*')

DATA_DIR = '../data/GraphDTA/data/davis'

with open(f'{DATA_DIR}/ligands_can.txt') as f:
    ligands = json.load(f)
with open(f'{DATA_DIR}/proteins.txt') as f:
    proteins = json.load(f)
with open(f'{DATA_DIR}/Y', 'rb') as f:
    Y = pickle.load(f, encoding='latin1')
with open(f'{DATA_DIR}/folds/train_fold_setting1.txt') as f:
    train_folds = json.load(f)
with open(f'{DATA_DIR}/folds/test_fold_setting1.txt') as f:
    test_fold = json.load(f)

train_indices = [idx for fold in train_folds for idx in fold]
ligand_ids = list(ligands.keys())
protein_names = list(proteins.keys())

ATOM_VOCAB = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'P', 'I']
AMINO_ACID_VOCAB = sorted(set(letter for seq in proteins.values() for letter in seq))
aa_to_index = {aa: i for i, aa in enumerate(AMINO_ACID_VOCAB)}

def one_hot(value, vocab):
    vec = [0] * (len(vocab) + 1)
    if value in vocab:
        vec[vocab.index(value)] = 1
    else:
        vec[-1] = 1
    return vec

def atom_features(atom):
    features = []
    features += one_hot(atom.GetSymbol(), ATOM_VOCAB)
    features += [atom.GetDegree()]
    features += [atom.GetFormalCharge()]
    features += [int(atom.GetIsAromatic())]
    features += [int(atom.IsInRing())]
    return features

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    all_atom_features = [atom_features(atom) for atom in mol.GetAtoms()]
    edge_sources, edge_targets = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_sources += [i, j]
        edge_targets += [j, i]
    x = torch.tensor(all_atom_features, dtype=torch.float)
    edge_index = torch.tensor([edge_sources, edge_targets], dtype=torch.long)
    return x, edge_index

def sequence_to_indices(sequence):
    return torch.tensor([aa_to_index[letter] for letter in sequence], dtype=torch.long)

def get_example(flat_index):
    drug_row, protein_col = np.unravel_index(flat_index, Y.shape)
    drug_id = ligand_ids[drug_row]
    smiles = ligands[drug_id]
    protein_name = protein_names[protein_col]
    sequence = proteins[protein_name]
    raw_kd_nM = Y[drug_row, protein_col]
    pKd = -np.log10(raw_kd_nM / 1e9)
    drug_x, drug_edge_index = smiles_to_graph(smiles)
    protein_indices = sequence_to_indices(sequence)
    return drug_x, drug_edge_index, protein_indices, pKd, drug_id, protein_name

def concordance_index(y_true, y_pred, sample_size=2000, seed=42):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)
    rng = np.random.default_rng(seed)
    concordant, discordant = 0, 0
    for _ in range(sample_size):
        i, j = rng.choice(n, size=2, replace=False)
        if y_true[i] == y_true[j]:
            continue
        if (y_true[i] > y_true[j]) == (y_pred[i] > y_pred[j]):
            concordant += 1
        else:
            discordant += 1
    return concordant / (concordant + discordant)

print('Drugs:', len(ligands), '| Proteins:', len(proteins), '| Train:', len(train_indices), '| Test:', len(test_fold))

Drugs: 68 | Proteins: 442 | Train: 25046 | Test: 5010


In [2]:
class DrugEncoder(nn.Module):
    def __init__(self, atom_feature_dim=14, hidden_dim=64):
        super().__init__()
        self.conv1 = GCNConv(atom_feature_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.attention_scorer = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        h = self.conv3(h, edge_index)
        scores = F.softmax(self.attention_scorer(h), dim=0)
        return (scores * h).sum(dim=0)

class ProteinEncoder(nn.Module):
    def __init__(self, vocab_size=21, embed_dim=8, hidden_dim=64, kernel_size=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv = nn.Conv1d(embed_dim, hidden_dim, kernel_size=kernel_size)

    def forward(self, sequence_indices):
        embedded = self.embedding(sequence_indices)
        x = embedded.unsqueeze(0).transpose(1, 2)
        conv_out = self.conv(x)
        pooled = conv_out.max(dim=2).values
        return pooled.squeeze(0)

class DTIModel(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.drug_encoder = DrugEncoder(hidden_dim=hidden_dim)
        self.protein_encoder = ProteinEncoder(hidden_dim=hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, drug_x, drug_edge_index, protein_sequence_indices):
        drug_vec = self.drug_encoder(drug_x, drug_edge_index)
        protein_vec = self.protein_encoder(protein_sequence_indices)
        combined = torch.cat([drug_vec, protein_vec])
        return self.mlp(combined)

print('Model classes defined.')

Model classes defined.


In [3]:
import mlflow

seeds = [0, 1, 2, 3, 4]
all_results = []

mlflow.set_experiment("DTI_Davis_Prediction")

for seed in seeds:
    torch.manual_seed(seed)
    random.seed(seed)

    model = DTIModel(hidden_dim=64)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    seed_train_indices = train_indices.copy()

    start = time.time()
    for epoch in range(8):
        random.shuffle(seed_train_indices)
        for flat_idx in seed_train_indices:
            drug_x, drug_edge_index, protein_indices, true_pKd, _, _ = get_example(flat_idx)
            optimizer.zero_grad()
            predicted_pKd = model(drug_x, drug_edge_index, protein_indices)
            loss = loss_fn(predicted_pKd, torch.tensor([true_pKd], dtype=torch.float))
            loss.backward()
            optimizer.step()

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for flat_idx in test_fold:
            drug_x, drug_edge_index, protein_indices, true_pKd, _, _ = get_example(flat_idx)
            preds.append(model(drug_x, drug_edge_index, protein_indices).item())
            trues.append(true_pKd)

    preds, trues = np.array(preds), np.array(trues)
    seed_r, _ = pearsonr(preds, trues)
    seed_mse = np.mean((preds - trues) ** 2)
    seed_ci = concordance_index(trues, preds, seed=seed)
    is_ceil = np.abs(trues - 5.0) < 0.001
    seed_real_r, _ = pearsonr(preds[~is_ceil], trues[~is_ceil])
    seed_real_mse = np.mean((preds[~is_ceil] - trues[~is_ceil]) ** 2)

    elapsed = (time.time() - start) / 60
    all_results.append({'seed': seed, 'r': seed_r, 'mse': seed_mse, 'ci': seed_ci,
                         'real_binder_r': seed_real_r, 'real_binder_mse': seed_real_mse})

    with mlflow.start_run(run_name=f"GNN_64dim_8epoch_seed{seed}"):
        mlflow.log_param("hidden_dim", 64)
        mlflow.log_param("num_epochs", 8)
        mlflow.log_param("seed", seed)
        mlflow.log_param("model_type", "GNN")
        mlflow.log_metric("overall_pearson_r", seed_r)
        mlflow.log_metric("overall_mse", seed_mse)
        mlflow.log_metric("real_binder_r", seed_real_r)
        mlflow.log_metric("real_binder_mse", seed_real_mse)
        mlflow.log_metric("concordance_index", seed_ci)

    print(f'Seed {seed}: R={seed_r:.4f} | CI={seed_ci:.4f} | real_binder_R={seed_real_r:.4f} | time={elapsed:.1f}min')

rs = [x['r'] for x in all_results]
cis = [x['ci'] for x in all_results]
real_rs = [x['real_binder_r'] for x in all_results]

print(f'\n=== Multi-seed summary (n={len(seeds)}) ===')
print(f'Pearson R:        {np.mean(rs):.4f} ± {np.std(rs):.4f}')
print(f'Concordance Index: {np.mean(cis):.4f} ± {np.std(cis):.4f}')
print(f'Real-binder R:     {np.mean(real_rs):.4f} ± {np.std(real_rs):.4f}')

Seed 0: R=0.5029 | CI=0.7626 | real_binder_R=0.3382 | time=23.5min
Seed 1: R=0.5479 | CI=0.7864 | real_binder_R=0.3437 | time=24.5min
Seed 2: R=0.4840 | CI=0.7458 | real_binder_R=0.3091 | time=23.7min
Seed 3: R=0.3207 | CI=0.6320 | real_binder_R=0.2367 | time=24.6min
Seed 4: R=0.5055 | CI=0.7600 | real_binder_R=0.3026 | time=23.6min

=== Multi-seed summary (n=5) ===
Pearson R:        0.4722 ± 0.0786
Concordance Index: 0.7374 ± 0.0543
Real-binder R:     0.3061 ± 0.0382


In [5]:
from rdkit.Chem import AllChem
import xgboost as xgb

def get_ecfp(smiles, n_bits=1024, radius=2):
    mol = Chem.MolFromSmiles(smiles)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

def get_aa_composition(sequence, vocab=AMINO_ACID_VOCAB):
    counts = np.array([sequence.count(aa) for aa in vocab], dtype=float)
    return counts / len(sequence)

def build_baseline_dataset(indices):
    X_list, y_list = [], []
    for flat_idx in indices:
        drug_row, protein_col = np.unravel_index(flat_idx, Y.shape)
        smiles = ligands[ligand_ids[drug_row]]
        sequence = proteins[protein_names[protein_col]]
        ecfp = get_ecfp(smiles)
        comp = get_aa_composition(sequence)
        raw_kd = Y[drug_row, protein_col]
        pKd = -np.log10(raw_kd / 1e9)
        X_list.append(np.concatenate([ecfp, comp]))
        y_list.append(pKd)
    return np.array(X_list), np.array(y_list)

print('Building training set...')
X_train, y_train = build_baseline_dataset(train_indices)
print('Building test set...')
X_test, y_test = build_baseline_dataset(test_fold)

xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_predictions = xgb_model.predict(X_test)

print('XGBoost rebuilt. Test shape:', X_test.shape)

Building training set...
Building test set...
XGBoost rebuilt. Test shape: (5010, 1045)


In [6]:
# Use our existing 5-seed GNN results (seed 1, our best run) and XGBoost results
# Re-run seed 1's model predictions, or reuse if still in memory as `preds`/`trues` from last seed (seed 4)
# To be rigorous, let's regenerate predictions explicitly tagged by model

BINDER_THRESHOLD = 7.0

def precision_recall_at_threshold(true_pKd, predicted_pKd, threshold, top_k_fractions):
    true_pKd = np.array(true_pKd)
    predicted_pKd = np.array(predicted_pKd)
    is_true_binder = true_pKd >= threshold

    print(f'Total test pairs: {len(true_pKd)}')
    print(f'True binders (pKd >= {threshold}): {is_true_binder.sum()} ({100*is_true_binder.mean():.2f}%)')

    # Rank all pairs by predicted affinity, descending
    order = np.argsort(-predicted_pKd)
    sorted_true_binder = is_true_binder[order]

    results = []
    for frac in top_k_fractions:
        k = int(len(true_pKd) * frac)
        top_k = sorted_true_binder[:k]
        precision = top_k.sum() / k
        recall = top_k.sum() / is_true_binder.sum()
        results.append({'top_fraction': frac, 'k': k, 'precision': precision, 'recall': recall})
        print(f'Top {frac*100:.0f}% ({k} pairs): precision={precision:.4f} | recall={recall:.4f}')
    return results

print("=== XGBoost ===")
xgb_pr = precision_recall_at_threshold(y_test, xgb_predictions, BINDER_THRESHOLD, [0.05, 0.10, 0.20])

=== XGBoost ===
Total test pairs: 5010
True binders (pKd >= 7.0): 417 (8.32%)
Top 5% (250 pairs): precision=0.6600 | recall=0.3957
Top 10% (501 pairs): precision=0.4850 | recall=0.5827
Top 20% (1002 pairs): precision=0.3263 | recall=0.7842


In [7]:
gnn_test_predictions_per_seed = []

for seed in [0, 1, 2, 3, 4]:
    torch.manual_seed(seed)
    random.seed(seed)
    model = DTIModel(hidden_dim=64)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    seed_train_indices = train_indices.copy()

    for epoch in range(8):
        random.shuffle(seed_train_indices)
        for flat_idx in seed_train_indices:
            drug_x, drug_edge_index, protein_indices, true_pKd, _, _ = get_example(flat_idx)
            optimizer.zero_grad()
            predicted_pKd = model(drug_x, drug_edge_index, protein_indices)
            loss = loss_fn(predicted_pKd, torch.tensor([true_pKd], dtype=torch.float))
            loss.backward()
            optimizer.step()

    model.eval()
    preds = []
    with torch.no_grad():
        for flat_idx in test_fold:
            drug_x, drug_edge_index, protein_indices, _, _, _ = get_example(flat_idx)
            preds.append(model(drug_x, drug_edge_index, protein_indices).item())
    gnn_test_predictions_per_seed.append(preds)
    print(f'Seed {seed} done')

gnn_ensemble_predictions = np.mean(gnn_test_predictions_per_seed, axis=0)
print('\nEnsemble predictions ready.')

Seed 0 done
Seed 1 done
Seed 2 done
Seed 3 done
Seed 4 done

Ensemble predictions ready.


In [1]:
print("=== GNN (5-seed ensemble average) ===")
gnn_pr = precision_recall_at_threshold(y_test, gnn_ensemble_predictions, BINDER_THRESHOLD, [0.05, 0.10, 0.20])

=== GNN (5-seed ensemble average) ===


NameError: name 'precision_recall_at_threshold' is not defined

In [3]:
import os
os.makedirs('../models', exist_ok=True)
print('Models folder ready.')

Models folder ready.


In [4]:
import torch

# After training each seed's model, save it
torch.save(model.state_dict(), f'../models/gnn_seed{seed}.pt')

# And save predictions/results as plain files too
import json
with open('../models/multi_seed_results.json', 'w') as f:
    json.dump(all_results, f)

NameError: name 'model' is not defined

In [5]:
import json
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import time
from rdkit import Chem
from rdkit import RDLogger
from torch_geometric.nn import GCNConv
from scipy.stats import pearsonr

RDLogger.DisableLog('rdApp.*')

DATA_DIR = '../data/GraphDTA/data/davis'

with open(f'{DATA_DIR}/ligands_can.txt') as f:
    ligands = json.load(f)
with open(f'{DATA_DIR}/proteins.txt') as f:
    proteins = json.load(f)
with open(f'{DATA_DIR}/Y', 'rb') as f:
    Y = pickle.load(f, encoding='latin1')
with open(f'{DATA_DIR}/folds/train_fold_setting1.txt') as f:
    train_folds = json.load(f)
with open(f'{DATA_DIR}/folds/test_fold_setting1.txt') as f:
    test_fold = json.load(f)

train_indices = [idx for fold in train_folds for idx in fold]
ligand_ids = list(ligands.keys())
protein_names = list(proteins.keys())

ATOM_VOCAB = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'P', 'I']
AMINO_ACID_VOCAB = sorted(set(letter for seq in proteins.values() for letter in seq))
aa_to_index = {aa: i for i, aa in enumerate(AMINO_ACID_VOCAB)}

def one_hot(value, vocab):
    vec = [0] * (len(vocab) + 1)
    if value in vocab:
        vec[vocab.index(value)] = 1
    else:
        vec[-1] = 1
    return vec

def atom_features(atom):
    features = []
    features += one_hot(atom.GetSymbol(), ATOM_VOCAB)
    features += [atom.GetDegree()]
    features += [atom.GetFormalCharge()]
    features += [int(atom.GetIsAromatic())]
    features += [int(atom.IsInRing())]
    return features

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    all_atom_features = [atom_features(atom) for atom in mol.GetAtoms()]
    edge_sources, edge_targets = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_sources += [i, j]
        edge_targets += [j, i]
    x = torch.tensor(all_atom_features, dtype=torch.float)
    edge_index = torch.tensor([edge_sources, edge_targets], dtype=torch.long)
    return x, edge_index

def sequence_to_indices(sequence):
    return torch.tensor([aa_to_index[letter] for letter in sequence], dtype=torch.long)

def get_example(flat_index):
    drug_row, protein_col = np.unravel_index(flat_index, Y.shape)
    drug_id = ligand_ids[drug_row]
    smiles = ligands[drug_id]
    protein_name = protein_names[protein_col]
    sequence = proteins[protein_name]
    raw_kd_nM = Y[drug_row, protein_col]
    pKd = -np.log10(raw_kd_nM / 1e9)
    drug_x, drug_edge_index = smiles_to_graph(smiles)
    protein_indices = sequence_to_indices(sequence)
    return drug_x, drug_edge_index, protein_indices, pKd, drug_id, protein_name

def concordance_index(y_true, y_pred, sample_size=2000, seed=42):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)
    rng = np.random.default_rng(seed)
    concordant, discordant = 0, 0
    for _ in range(sample_size):
        i, j = rng.choice(n, size=2, replace=False)
        if y_true[i] == y_true[j]:
            continue
        if (y_true[i] > y_true[j]) == (y_pred[i] > y_pred[j]):
            concordant += 1
        else:
            discordant += 1
    return concordant / (concordant + discordant)

print('Drugs:', len(ligands), '| Proteins:', len(proteins), '| Train:', len(train_indices), '| Test:', len(test_fold))

Drugs: 68 | Proteins: 442 | Train: 25046 | Test: 5010


In [6]:
class DrugEncoder(nn.Module):
    def __init__(self, atom_feature_dim=14, hidden_dim=64):
        super().__init__()
        self.conv1 = GCNConv(atom_feature_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.attention_scorer = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        h = self.conv3(h, edge_index)
        scores = F.softmax(self.attention_scorer(h), dim=0)
        return (scores * h).sum(dim=0)

class ProteinEncoder(nn.Module):
    def __init__(self, vocab_size=21, embed_dim=8, hidden_dim=64, kernel_size=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv = nn.Conv1d(embed_dim, hidden_dim, kernel_size=kernel_size)

    def forward(self, sequence_indices):
        embedded = self.embedding(sequence_indices)
        x = embedded.unsqueeze(0).transpose(1, 2)
        conv_out = self.conv(x)
        pooled = conv_out.max(dim=2).values
        return pooled.squeeze(0)

class DTIModel(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.drug_encoder = DrugEncoder(hidden_dim=hidden_dim)
        self.protein_encoder = ProteinEncoder(hidden_dim=hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, drug_x, drug_edge_index, protein_sequence_indices):
        drug_vec = self.drug_encoder(drug_x, drug_edge_index)
        protein_vec = self.protein_encoder(protein_sequence_indices)
        combined = torch.cat([drug_vec, protein_vec])
        return self.mlp(combined)

print('Model classes defined.')

Model classes defined.


In [7]:
import os
import mlflow

seeds = [0, 1, 2, 3, 4]
all_results = []
all_predictions = []

mlflow.set_experiment("DTI_Davis_Prediction")

# Resume support: skip seeds already completed
completed_seeds = set()
if os.path.exists('../models/multi_seed_results.json'):
    with open('../models/multi_seed_results.json') as f:
        all_results = json.load(f)
    completed_seeds = set(r['seed'] for r in all_results)
    print(f'Resuming - already completed seeds: {sorted(completed_seeds)}')

for seed in seeds:
    if seed in completed_seeds:
        print(f'Seed {seed} already done, skipping.')
        continue

    torch.manual_seed(seed)
    random.seed(seed)

    model = DTIModel(hidden_dim=64)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    seed_train_indices = train_indices.copy()
    start = time.time()

    for epoch in range(8):
        random.shuffle(seed_train_indices)
        for flat_idx in seed_train_indices:
            drug_x, drug_edge_index, protein_indices, true_pKd, _, _ = get_example(flat_idx)
            optimizer.zero_grad()
            predicted_pKd = model(drug_x, drug_edge_index, protein_indices)
            loss = loss_fn(predicted_pKd, torch.tensor([true_pKd], dtype=torch.float))
            loss.backward()
            optimizer.step()

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for flat_idx in test_fold:
            drug_x, drug_edge_index, protein_indices, true_pKd, _, _ = get_example(flat_idx)
            preds.append(model(drug_x, drug_edge_index, protein_indices).item())
            trues.append(true_pKd)

    preds_arr, trues_arr = np.array(preds), np.array(trues)
    seed_r, _ = pearsonr(preds_arr, trues_arr)
    seed_mse = np.mean((preds_arr - trues_arr) ** 2)
    seed_ci = concordance_index(trues_arr, preds_arr, seed=seed)
    is_ceil = np.abs(trues_arr - 5.0) < 0.001
    seed_real_r, _ = pearsonr(preds_arr[~is_ceil], trues_arr[~is_ceil])
    seed_real_mse = np.mean((preds_arr[~is_ceil] - trues_arr[~is_ceil]) ** 2)
    elapsed = (time.time() - start) / 60

    # SAVE IMMEDIATELY - model weights + predictions + results
    torch.save(model.state_dict(), f'../models/gnn_seed{seed}.pt')
    with open(f'../models/gnn_seed{seed}_predictions.json', 'w') as f:
        json.dump({'predictions': preds, 'true_values': trues}, f)

    all_results.append({'seed': seed, 'r': seed_r, 'mse': seed_mse, 'ci': seed_ci,
                         'real_binder_r': seed_real_r, 'real_binder_mse': seed_real_mse})
    with open('../models/multi_seed_results.json', 'w') as f:
        json.dump(all_results, f)

    with mlflow.start_run(run_name=f"GNN_64dim_8epoch_seed{seed}_checkpointed"):
        mlflow.log_param("hidden_dim", 64)
        mlflow.log_param("num_epochs", 8)
        mlflow.log_param("seed", seed)
        mlflow.log_metric("overall_pearson_r", seed_r)
        mlflow.log_metric("concordance_index", seed_ci)
        mlflow.log_metric("real_binder_r", seed_real_r)

    print(f'Seed {seed}: R={seed_r:.4f} | CI={seed_ci:.4f} | real_binder_R={seed_real_r:.4f} | time={elapsed:.1f}min | SAVED to disk')

print('\nAll seeds complete and saved to disk.')

Seed 0: R=0.5029 | CI=0.7626 | real_binder_R=0.3382 | time=23.0min | SAVED to disk
Seed 1: R=0.5479 | CI=0.7864 | real_binder_R=0.3437 | time=22.7min | SAVED to disk
Seed 2: R=0.4840 | CI=0.7458 | real_binder_R=0.3091 | time=25.6min | SAVED to disk
Seed 3: R=0.3207 | CI=0.6320 | real_binder_R=0.2367 | time=23.7min | SAVED to disk
Seed 4: R=0.5055 | CI=0.7600 | real_binder_R=0.3026 | time=22.0min | SAVED to disk

All seeds complete and saved to disk.


In [8]:
gnn_test_predictions_per_seed = []

for seed in [0, 1, 2, 3, 4]:
    with open(f'../models/gnn_seed{seed}_predictions.json') as f:
        data = json.load(f)
    gnn_test_predictions_per_seed.append(data['predictions'])

gnn_ensemble_predictions = np.mean(gnn_test_predictions_per_seed, axis=0)
true_pKds_from_disk = np.array(data['true_values'])  # same true values every seed, just grab the last one

print('Ensemble predictions loaded from disk. Shape:', gnn_ensemble_predictions.shape)

Ensemble predictions loaded from disk. Shape: (5010,)


In [9]:
BINDER_THRESHOLD = 7.0

def precision_recall_at_threshold(true_pKd, predicted_pKd, threshold, top_k_fractions):
    true_pKd = np.array(true_pKd)
    predicted_pKd = np.array(predicted_pKd)
    is_true_binder = true_pKd >= threshold
    print(f'Total test pairs: {len(true_pKd)}')
    print(f'True binders (pKd >= {threshold}): {is_true_binder.sum()} ({100*is_true_binder.mean():.2f}%)')
    order = np.argsort(-predicted_pKd)
    sorted_true_binder = is_true_binder[order]
    results = []
    for frac in top_k_fractions:
        k = int(len(true_pKd) * frac)
        top_k = sorted_true_binder[:k]
        precision = top_k.sum() / k
        recall = top_k.sum() / is_true_binder.sum()
        results.append({'top_fraction': frac, 'precision': precision, 'recall': recall})
        print(f'Top {frac*100:.0f}% ({k} pairs): precision={precision:.4f} | recall={recall:.4f}')
    return results

print("=== GNN (5-seed ensemble) ===")
gnn_pr = precision_recall_at_threshold(true_pKds_from_disk, gnn_ensemble_predictions, BINDER_THRESHOLD, [0.05, 0.10, 0.20])

=== GNN (5-seed ensemble) ===
Total test pairs: 5010
True binders (pKd >= 7.0): 417 (8.32%)
Top 5% (250 pairs): precision=0.5200 | recall=0.3118
Top 10% (501 pairs): precision=0.3852 | recall=0.4628
Top 20% (1002 pairs): precision=0.2784 | recall=0.6691


In [10]:
import re

def parse_mutation(protein_name):
    """Extract (position, original_aa, mutant_aa) from names like 'FLT3(D835H)'.
    Returns None if no mutation pattern found (wild-type protein)."""
    match = re.search(r'\((\D)(\d+)(\D)\)', protein_name)
    if match:
        original_aa, position, mutant_aa = match.group(1), int(match.group(2)), match.group(3)
        return position, original_aa, mutant_aa
    return None

# quick test against real Davis names
test_names = ['FLT3', 'FLT3(D835H)', 'ABL1(T315I)', 'EGFR(L858R)', 'PIK3CA(H1047L)', 'JAK1(JH1domain-catalytic)']
for name in test_names:
    result = parse_mutation(name)
    print(f'{name:30s} -> {result}')

FLT3                           -> None
FLT3(D835H)                    -> (835, 'D', 'H')
ABL1(T315I)                    -> (315, 'T', 'I')
EGFR(L858R)                    -> (858, 'L', 'R')
PIK3CA(H1047L)                 -> (1047, 'H', 'L')
JAK1(JH1domain-catalytic)      -> None


In [11]:
mutation_count = 0
wild_type_count = 0
for name in protein_names:
    if parse_mutation(name) is not None:
        mutation_count += 1
    else:
        wild_type_count += 1

print(f'Proteins with parseable point mutations: {mutation_count}')
print(f'Proteins without (wild-type or non-mutation labels): {wild_type_count}')
print(f'Total: {mutation_count + wild_type_count}')

Proteins with parseable point mutations: 45
Proteins without (wild-type or non-mutation labels): 397
Total: 442


In [12]:
AA_LIST = sorted(set(AMINO_ACID_VOCAB))  # reuse our existing 21-letter vocab

def mutation_features(protein_name, max_seq_length=2500):
    """Returns a feature vector encoding mutation info, or all-zeros if wild-type."""
    result = parse_mutation(protein_name)
    
    has_mutation = [0]
    original_aa_onehot = [0] * len(AA_LIST)
    mutant_aa_onehot = [0] * len(AA_LIST)
    normalized_position = [0.0]
    
    if result is not None:
        position, original_aa, mutant_aa = result
        has_mutation = [1]
        if original_aa in AA_LIST:
            original_aa_onehot[AA_LIST.index(original_aa)] = 1
        if mutant_aa in AA_LIST:
            mutant_aa_onehot[AA_LIST.index(mutant_aa)] = 1
        normalized_position = [position / max_seq_length]  # scale roughly to 0-1 range
    
    return has_mutation + original_aa_onehot + mutant_aa_onehot + normalized_position

# test on real examples
for name in ['FLT3', 'FLT3(D835H)', 'ABL1(T315I)']:
    feats = mutation_features(name)
    print(f'{name:20s} -> length={len(feats)} | has_mutation={feats[0]} | position_norm={feats[-1]:.4f}')

FLT3                 -> length=44 | has_mutation=0 | position_norm=0.0000
FLT3(D835H)          -> length=44 | has_mutation=1 | position_norm=0.3340
ABL1(T315I)          -> length=44 | has_mutation=1 | position_norm=0.1260


In [15]:
def get_aa_composition_with_mutation(sequence, protein_name, vocab=AMINO_ACID_VOCAB):
    counts = np.array([sequence.count(aa) for aa in vocab], dtype=float)
    composition = counts / len(sequence)
    mut_feats = np.array(mutation_features(protein_name), dtype=float)
    return np.concatenate([composition, mut_feats])

def build_baseline_dataset_v2(indices):
    X_list, y_list = [], []
    for flat_idx in indices:
        drug_row, protein_col = np.unravel_index(flat_idx, Y.shape)
        smiles = ligands[ligand_ids[drug_row]]
        protein_name = protein_names[protein_col]
        sequence = proteins[protein_name]
        ecfp = get_ecfp(smiles)
        comp_mut = get_aa_composition_with_mutation(sequence, protein_name)
        raw_kd = Y[drug_row, protein_col]
        pKd = -np.log10(raw_kd / 1e9)
        X_list.append(np.concatenate([ecfp, comp_mut]))
        y_list.append(pKd)
    return np.array(X_list), np.array(y_list)

print('Building v2 training set (with mutation features)...')
X_train_v2, y_train_v2 = build_baseline_dataset_v2(train_indices)
print('Building v2 test set...')
X_test_v2, y_test_v2 = build_baseline_dataset_v2(test_fold)
print('Shapes:', X_train_v2.shape, X_test_v2.shape)

xgb_model_v2 = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
xgb_model_v2.fit(X_train_v2, y_train_v2)

print('v2 model trained.')

Building v2 training set (with mutation features)...
Building v2 test set...
Shapes: (25046, 1089) (5010, 1089)
v2 model trained.


In [14]:
from rdkit.Chem import AllChem
import xgboost as xgb

def get_ecfp(smiles, n_bits=1024, radius=2):
    mol = Chem.MolFromSmiles(smiles)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

print('get_ecfp defined.')

get_ecfp defined.


In [16]:
loratadine_smiles = 'CCOC(=O)N1CCC(=C2c3cc(Cl)ccc3CCc4cccnc24)CC1'
flt3_variants = ['FLT3', 'FLT3(D835H)', 'FLT3(D835Y)', 'FLT3(ITD)', 'FLT3(K663Q)', 'FLT3(N841I)', 'FLT3(R834Q)']

ecfp = get_ecfp(loratadine_smiles)

print('Loratadine vs FLT3 variants, WITH mutation features:')
for variant in flt3_variants:
    if variant in protein_names:
        sequence = proteins[variant]
        comp_mut = get_aa_composition_with_mutation(sequence, variant)
        features = np.concatenate([ecfp, comp_mut]).reshape(1, -1)
        pred = xgb_model_v2.predict(features)[0]
        print(f'{variant:20s} -> predicted pKd = {pred:.6f}')
    else:
        print(f'{variant:20s} -> not in Davis')

Loratadine vs FLT3 variants, WITH mutation features:
FLT3                 -> predicted pKd = 6.637527
FLT3(D835H)          -> predicted pKd = 7.069845
FLT3(D835Y)          -> predicted pKd = 7.069845
FLT3(ITD)            -> predicted pKd = 6.637527
FLT3(K663Q)          -> predicted pKd = 7.107318
FLT3(N841I)          -> predicted pKd = 7.047935
FLT3(R834Q)          -> predicted pKd = 7.035396


In [17]:
from scipy.stats import pearsonr

xgb_predictions_v2 = xgb_model_v2.predict(X_test_v2)

r_xgb_v2, _ = pearsonr(xgb_predictions_v2, y_test_v2)
mse_xgb_v2 = np.mean((xgb_predictions_v2 - y_test_v2) ** 2)

is_ceiling_v2 = np.abs(y_test_v2 - 5.0) < 0.001
real_binder_r_xgb_v2, _ = pearsonr(xgb_predictions_v2[~is_ceiling_v2], y_test_v2[~is_ceiling_v2])
real_binder_mse_xgb_v2 = np.mean((xgb_predictions_v2[~is_ceiling_v2] - y_test_v2[~is_ceiling_v2]) ** 2)

ci_xgb_v2 = concordance_index(y_test_v2, xgb_predictions_v2)

print('=== XGBoost v2 (with mutation features) ===')
print(f'Overall R: {r_xgb_v2:.4f} | Real-binder R: {real_binder_r_xgb_v2:.4f} | CI: {ci_xgb_v2:.4f}')
print(f'Overall MSE: {mse_xgb_v2:.4f} | Real-binder MSE: {real_binder_mse_xgb_v2:.4f}')

print('\n=== XGBoost v1 (original, no mutation features) ===')
print(f'Overall R: {r_xgb:.4f} | Real-binder R: {real_binder_r_xgb:.4f} | CI: {ci_xgb:.4f}')
print(f'Overall MSE: {mse_xgb:.4f} | Real-binder MSE: {real_binder_mse_xgb:.4f}')

=== XGBoost v2 (with mutation features) ===
Overall R: 0.7138 | Real-binder R: 0.5788 | CI: 0.8368
Overall MSE: 0.3990 | Real-binder MSE: 1.0131

=== XGBoost v1 (original, no mutation features) ===


NameError: name 'r_xgb' is not defined

In [19]:
def get_aa_composition(sequence, vocab=AMINO_ACID_VOCAB):
    counts = np.array([sequence.count(aa) for aa in vocab], dtype=float)
    return counts / len(sequence)

def build_baseline_dataset_v1(indices):
    X_list, y_list = [], []
    for flat_idx in indices:
        drug_row, protein_col = np.unravel_index(flat_idx, Y.shape)
        smiles = ligands[ligand_ids[drug_row]]
        sequence = proteins[protein_names[protein_col]]
        ecfp = get_ecfp(smiles)
        comp = get_aa_composition(sequence)
        raw_kd = Y[drug_row, protein_col]
        pKd = -np.log10(raw_kd / 1e9)
        X_list.append(np.concatenate([ecfp, comp]))
        y_list.append(pKd)
    return np.array(X_list), np.array(y_list)

print('Building v1 (no mutation features) train/test sets...')
X_train_v1, y_train_v1 = build_baseline_dataset_v1(train_indices)
X_test_v1, y_test_v1 = build_baseline_dataset_v1(test_fold)

xgb_model_v1 = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
xgb_model_v1.fit(X_train_v1, y_train_v1)
xgb_predictions_v1 = xgb_model_v1.predict(X_test_v1)

r_xgb_v1, _ = pearsonr(xgb_predictions_v1, y_test_v1)
is_ceiling_v1 = np.abs(y_test_v1 - 5.0) < 0.001
real_binder_r_xgb_v1, _ = pearsonr(xgb_predictions_v1[~is_ceiling_v1], y_test_v1[~is_ceiling_v1])
ci_xgb_v1 = concordance_index(y_test_v1, xgb_predictions_v1)

print(f'v1: Overall R={r_xgb_v1:.4f} | Real-binder R={real_binder_r_xgb_v1:.4f} | CI={ci_xgb_v1:.4f}')
print(f'v2: Overall R={r_xgb_v2:.4f} | Real-binder R={real_binder_r_xgb_v2:.4f} | CI={ci_xgb_v2:.4f}')

Building v1 (no mutation features) train/test sets...
v1: Overall R=0.7087 | Real-binder R=0.5632 | CI=0.8263
v2: Overall R=0.7138 | Real-binder R=0.5788 | CI=0.8368


In [20]:
import urllib.request

url = 'https://www.cureffi.org/wp-content/uploads/2013/10/drugs.txt'
with urllib.request.urlopen(url) as response:
    raw_text = response.read().decode('utf-8')

lines = raw_text.strip().split('\n')
header = lines[0].split('\t')
print('Columns:', header)

approved_drugs = []
for line in lines[1:]:
    parts = line.split('\t')
    if len(parts) == 3:
        name, cns_flag, smiles = parts
        if smiles.strip():  # skip biologics with empty SMILES
            approved_drugs.append({'name': name, 'cns_drug': cns_flag == 'TRUE', 'smiles': smiles})

print(f'Total approved drugs with valid SMILES: {len(approved_drugs)}')
print('\nFirst 5:')
for d in approved_drugs[:5]:
    print(d)

Columns: ['generic_name', 'cns_drug', 'smiles']
Total approved drugs with valid SMILES: 1497

First 5:
{'name': 'Abacavir', 'cns_drug': False, 'smiles': 'NC1=NC2=C(N=CN2[C@@H]2C[C@H](CO)C=C2)C(NC2CC2)=N1'}
{'name': 'Abiraterone', 'cns_drug': False, 'smiles': 'CC(=O)O[C@H]1CC[C@]2(C)C3CC[C@@]4(C)C(CC=C4C4=CN=CC=C4)C3CC=C2C1'}
{'name': 'Acamprosate', 'cns_drug': True, 'smiles': 'CC(=O)NCCCS(O)(=O)=O'}
{'name': 'Acarbose', 'cns_drug': False, 'smiles': 'C[C@H]1O[C@H](O[C@@H]2[C@@H](CO)O[C@H](O[C@@H]3[C@@H](CO)O[C@@H](O)[C@H](O)[C@H]3O)[C@H](O)[C@H]2O)[C@H](O)[C@@H](O)[C@@H]1N[C@H]1C=C(CO)[C@H](O)[C@H](O)[C@H]1O'}
{'name': 'Acebutolol', 'cns_drug': False, 'smiles': 'CCCC(=O)NC1=CC(C(C)=O)=C(OCC(O)CNC(C)C)C=C1'}


In [21]:
valid_drugs = []
invalid_count = 0

for d in approved_drugs:
    mol = Chem.MolFromSmiles(d['smiles'])
    if mol is not None and mol.GetNumAtoms() > 0:
        valid_drugs.append(d)
    else:
        invalid_count += 1

print(f'Valid (RDKit-parseable): {len(valid_drugs)}')
print(f'Invalid/unparseable: {invalid_count}')

Valid (RDKit-parseable): 1497
Invalid/unparseable: 0


In [23]:
known_oncology_kinases = [
    'EGFR', 'ABL1', 'BRAF', 'ALK', 'KIT', 'MET', 'RET', 'ROS1',
    'FLT3', 'JAK2', 'BCR', 'PDGFRA', 'PDGFRB', 'KDR', 'FGFR1',
    'FGFR2', 'FGFR3', 'SRC', 'AKT1', 'AKT2', 'MTOR', 'CDK4', 'CDK6',
    'PIK3CA', 'BTK', 'JAK1', 'JAK3', 'MAPK1', 'MAPK3', 'AURKA', 'AURKB'
]

found_in_davis = []
for name in known_oncology_kinases:
    matches = [p for p in protein_names if name in p]
    if matches:
        found_in_davis.append((name, matches))

oncology_panel = sorted(set(
    match for name, matches in found_in_davis for match in matches
))

print(f'Oncology panel rebuilt: {len(oncology_panel)} proteins')

Oncology panel rebuilt: 83 proteins


In [24]:
import time

start = time.time()
results = []

for i, drug in enumerate(valid_drugs):
    ecfp = get_ecfp(drug['smiles'])
    for protein_name in oncology_panel:
        sequence = proteins[protein_name]
        comp_mut = get_aa_composition_with_mutation(sequence, protein_name)
        features = np.concatenate([ecfp, comp_mut]).reshape(1, -1)
        predicted_pKd = xgb_model_v2.predict(features)[0]
        results.append((drug['name'], protein_name, predicted_pKd))
    if (i+1) % 200 == 0:
        elapsed = time.time() - start
        print(f'{i+1}/{len(valid_drugs)} drugs done | {elapsed:.0f}s elapsed')

elapsed = time.time() - start
print(f'\nTotal: {len(results)} predictions in {elapsed:.0f}s')

200/1497 drugs done | 7s elapsed
400/1497 drugs done | 15s elapsed
600/1497 drugs done | 22s elapsed
800/1497 drugs done | 29s elapsed
1000/1497 drugs done | 37s elapsed
1200/1497 drugs done | 45s elapsed
1400/1497 drugs done | 53s elapsed

Total: 124251 predictions in 57s


In [26]:
import pandas as pd
results_df = pd.DataFrame(results, columns=['drug', 'protein', 'predicted_pKd'])
results_df_sorted = results_df.sort_values('predicted_pKd', ascending=False)

print('Top 20 predicted drug-target pairs across the full screen:')
print(results_df_sorted.head(20).to_string(index=False))

print(f'\nNumber of predictions with pKd >= 7.0 (our "true binder" threshold): {(results_df["predicted_pKd"] >= 7.0).sum()}')
print(f'Number of unique drugs appearing in those: {results_df[results_df["predicted_pKd"] >= 7.0]["drug"].nunique()}')

Top 20 predicted drug-target pairs across the full screen:
     drug          protein  predicted_pKd
Dasatinib     ABL1(H396P)p       9.308064
Dasatinib      ABL1(H396P)       9.308064
Dasatinib      ABL1(F317L)       9.266291
Dasatinib     ABL1(F317L)p       9.266291
Bosutinib      ABL1(F317L)       9.264635
Bosutinib     ABL1(F317L)p       9.264635
Dasatinib     ABL1(F317I)p       9.099435
Dasatinib      ABL1(F317I)       9.099435
Sunitinib       KIT(V559D)       9.071085
Bosutinib     ABL1(H396P)p       8.999436
Bosutinib      ABL1(H396P)       8.999436
Bosutinib     ABL1(F317I)p       8.990345
Bosutinib      ABL1(F317I)       8.990345
Sunitinib       KIT(L576P)       8.931910
Dasatinib      ABL1(Y253F)       8.767842
Dasatinib     ABL1(Q252H)p       8.767842
Dasatinib      ABL1(E255K)       8.767842
Dasatinib      ABL1(Q252H)       8.767842
Dasatinib      ABL1(M351T)       8.760963
Sunitinib KIT(V559D-T670I)       8.692590

Number of predictions with pKd >= 7.0 (our "true binder" t

In [27]:
known_kinase_inhibitor_keywords = ['nib', 'tinib']  # most approved kinase inhibitors end in -nib/-tinib by INN naming convention

non_kinase_results = results_df_sorted[
    ~results_df_sorted['drug'].str.lower().str.contains('|'.join(known_kinase_inhibitor_keywords))
]

print('Top 15 predicted pairs EXCLUDING drugs with kinase-inhibitor naming conventions:')
print(non_kinase_results.head(15).to_string(index=False))

Top 15 predicted pairs EXCLUDING drugs with kinase-inhibitor naming conventions:
      drug      protein  predicted_pKd
 Josamycin  FLT3(N841I)       8.664656
 Josamycin  FLT3(D835Y)       8.599868
 Josamycin  FLT3(D835H)       8.599868
 Josamycin  FLT3(K663Q)       8.589418
 Josamycin         FLT3       8.413048
 Josamycin    FLT3(ITD)       8.413048
  Eribulin  FLT3(D835H)       8.362243
  Eribulin  FLT3(D835Y)       8.362243
  Eribulin  FLT3(K663Q)       8.350968
  Eribulin  FLT3(N841I)       8.303149
 Josamycin  FLT3(R834Q)       8.224710
Tacrolimus  FLT3(N841I)       8.223297
Nevirapine  ABL1(F317L)       8.178806
Nevirapine ABL1(F317L)p       8.178806
Tacrolimus  FLT3(K663Q)       8.175551


In [28]:
import random as rnd

rnd.seed(123)
all_drug_indices = list(range(68))
rnd.shuffle(all_drug_indices)

held_out_drugs = set(all_drug_indices[:14])   # ~20% of drugs held out entirely
train_drugs = set(all_drug_indices[14:])

print(f'Held-out drugs: {len(held_out_drugs)} | Train drugs: {len(train_drugs)}')

# Now split ALL valid pairs (not just the literature fold) by this drug-level rule
drug_holdout_train_indices = []
drug_holdout_test_indices = []

for flat_idx in range(Y.size):
    drug_row, protein_col = np.unravel_index(flat_idx, Y.shape)
    if np.isnan(Y[drug_row, protein_col]):
        continue
    if drug_row in held_out_drugs:
        drug_holdout_test_indices.append(flat_idx)
    else:
        drug_holdout_train_indices.append(flat_idx)

print(f'Drug-holdout train pairs: {len(drug_holdout_train_indices)}')
print(f'Drug-holdout test pairs: {len(drug_holdout_test_indices)}')

Held-out drugs: 14 | Train drugs: 54
Drug-holdout train pairs: 23868
Drug-holdout test pairs: 6188


In [29]:
print('Total Y.size (68*442):', Y.size)
print('Total valid (non-NaN) pairs:', np.sum(~np.isnan(Y)))
print('Sum of our two splits:', len(drug_holdout_train_indices) + len(drug_holdout_test_indices))

Total Y.size (68*442): 30056
Total valid (non-NaN) pairs: 30056
Sum of our two splits: 30056


In [30]:
def evaluate_on_indices(model, indices):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for flat_idx in indices:
            drug_x, drug_edge_index, protein_indices, true_pKd, _, _ = get_example(flat_idx)
            preds.append(model(drug_x, drug_edge_index, protein_indices).item())
            trues.append(true_pKd)
    return np.array(preds), np.array(trues)

drug_holdout_results = []

for seed in [0, 1, 2, 3, 4]:
    model = DTIModel(hidden_dim=64)
    model.load_state_dict(torch.load(f'../models/gnn_seed{seed}.pt'))

    preds, trues = evaluate_on_indices(model, drug_holdout_test_indices)
    r, _ = pearsonr(preds, trues)
    ci = concordance_index(trues, preds, seed=seed)

    drug_holdout_results.append({'seed': seed, 'r': r, 'ci': ci})
    print(f'Seed {seed}: drug-holdout R={r:.4f} | CI={ci:.4f}')

rs = [x['r'] for x in drug_holdout_results]
cis = [x['ci'] for x in drug_holdout_results]
print(f'\nDrug-holdout mean R: {np.mean(rs):.4f} ± {np.std(rs):.4f}')
print(f'Drug-holdout mean CI: {np.mean(cis):.4f} ± {np.std(cis):.4f}')

Seed 0: drug-holdout R=0.5601 | CI=0.7823
Seed 1: drug-holdout R=0.5592 | CI=0.7971
Seed 2: drug-holdout R=0.4765 | CI=0.7386
Seed 3: drug-holdout R=0.3215 | CI=0.6900
Seed 4: drug-holdout R=0.5091 | CI=0.7681

Drug-holdout mean R: 0.4853 ± 0.0878
Drug-holdout mean CI: 0.7552 ± 0.0379


In [31]:
rnd.seed(456)
all_protein_indices = list(range(442))
rnd.shuffle(all_protein_indices)

held_out_proteins = set(all_protein_indices[:88])   # ~20% of proteins held out entirely
train_proteins = set(all_protein_indices[88:])

print(f'Held-out proteins: {len(held_out_proteins)} | Train proteins: {len(train_proteins)}')

protein_holdout_train_indices = []
protein_holdout_test_indices = []

for flat_idx in range(Y.size):
    drug_row, protein_col = np.unravel_index(flat_idx, Y.shape)
    if np.isnan(Y[drug_row, protein_col]):
        continue
    if protein_col in held_out_proteins:
        protein_holdout_test_indices.append(flat_idx)
    else:
        protein_holdout_train_indices.append(flat_idx)

print(f'Protein-holdout train pairs: {len(protein_holdout_train_indices)}')
print(f'Protein-holdout test pairs: {len(protein_holdout_test_indices)}')
print(f'Sanity check sum: {len(protein_holdout_train_indices) + len(protein_holdout_test_indices)} (should be 30056)')

Held-out proteins: 88 | Train proteins: 354
Protein-holdout train pairs: 24072
Protein-holdout test pairs: 5984
Sanity check sum: 30056 (should be 30056)


In [32]:
protein_holdout_results = []

for seed in [0, 1, 2, 3, 4]:
    model = DTIModel(hidden_dim=64)
    model.load_state_dict(torch.load(f'../models/gnn_seed{seed}.pt'))

    preds, trues = evaluate_on_indices(model, protein_holdout_test_indices)
    r, _ = pearsonr(preds, trues)
    ci = concordance_index(trues, preds, seed=seed)

    protein_holdout_results.append({'seed': seed, 'r': r, 'ci': ci})
    print(f'Seed {seed}: protein-holdout R={r:.4f} | CI={ci:.4f}')

rs = [x['r'] for x in protein_holdout_results]
cis = [x['ci'] for x in protein_holdout_results]
print(f'\nProtein-holdout mean R: {np.mean(rs):.4f} ± {np.std(rs):.4f}')
print(f'Protein-holdout mean CI: {np.mean(cis):.4f} ± {np.std(cis):.4f}')

Seed 0: protein-holdout R=0.5226 | CI=0.7673
Seed 1: protein-holdout R=0.5586 | CI=0.7901
Seed 2: protein-holdout R=0.5107 | CI=0.7400
Seed 3: protein-holdout R=0.2975 | CI=0.6554
Seed 4: protein-holdout R=0.5312 | CI=0.7778

Protein-holdout mean R: 0.4841 ± 0.0946
Protein-holdout mean CI: 0.7461 ± 0.0483


In [33]:
def parse_mutation_v2(protein_name):
    """Improved parser: handles point mutations AND insertion/duplication-type mutations."""
    # Point mutation: e.g. D835H
    point_match = re.search(r'\((\D)(\d+)(\D)\)', protein_name)
    if point_match:
        original_aa, position, mutant_aa = point_match.group(1), int(point_match.group(2)), point_match.group(3)
        return {'type': 'point', 'position': position, 'original_aa': original_aa, 'mutant_aa': mutant_aa}
    
    # Insertion/duplication: e.g. ITD (no specific position given in Davis naming)
    if 'ITD' in protein_name or 'insertion' in protein_name.lower() or 'dup' in protein_name.lower():
        return {'type': 'insertion', 'position': None, 'original_aa': None, 'mutant_aa': None}
    
    return None

def mutation_features_v2(protein_name, max_seq_length=2500):
    result = parse_mutation_v2(protein_name)
    
    is_point_mutation = [0]
    is_insertion_mutation = [0]
    original_aa_onehot = [0] * len(AA_LIST)
    mutant_aa_onehot = [0] * len(AA_LIST)
    normalized_position = [0.0]
    
    if result is not None:
        if result['type'] == 'point':
            is_point_mutation = [1]
            if result['original_aa'] in AA_LIST:
                original_aa_onehot[AA_LIST.index(result['original_aa'])] = 1
            if result['mutant_aa'] in AA_LIST:
                mutant_aa_onehot[AA_LIST.index(result['mutant_aa'])] = 1
            normalized_position = [result['position'] / max_seq_length]
        elif result['type'] == 'insertion':
            is_insertion_mutation = [1]
    
    return is_point_mutation + is_insertion_mutation + original_aa_onehot + mutant_aa_onehot + normalized_position

# Test: does ITD now get correctly flagged as a real mutation, distinct from wild-type?
for name in ['FLT3', 'FLT3(ITD)', 'FLT3(D835H)', 'FLT3(D835Y)']:
    feats = mutation_features_v2(name)
    print(f'{name:20s} -> is_point={feats[0]} | is_insertion={feats[1]} | position={feats[-1]:.4f}')

FLT3                 -> is_point=0 | is_insertion=0 | position=0.0000
FLT3(ITD)            -> is_point=0 | is_insertion=1 | position=0.0000
FLT3(D835H)          -> is_point=1 | is_insertion=0 | position=0.3340
FLT3(D835Y)          -> is_point=1 | is_insertion=0 | position=0.3340


In [34]:
def get_aa_composition_with_mutation_v2(sequence, protein_name, vocab=AMINO_ACID_VOCAB):
    counts = np.array([sequence.count(aa) for aa in vocab], dtype=float)
    composition = counts / len(sequence)
    mut_feats = np.array(mutation_features_v2(protein_name), dtype=float)
    return np.concatenate([composition, mut_feats])

def build_baseline_dataset_v3(indices):
    X_list, y_list = [], []
    for flat_idx in indices:
        drug_row, protein_col = np.unravel_index(flat_idx, Y.shape)
        smiles = ligands[ligand_ids[drug_row]]
        protein_name = protein_names[protein_col]
        sequence = proteins[protein_name]
        ecfp = get_ecfp(smiles)
        comp_mut = get_aa_composition_with_mutation_v2(sequence, protein_name)
        raw_kd = Y[drug_row, protein_col]
        pKd = -np.log10(raw_kd / 1e9)
        X_list.append(np.concatenate([ecfp, comp_mut]))
        y_list.append(pKd)
    return np.array(X_list), np.array(y_list)

print('Building v3 (ITD-aware) train/test sets...')
X_train_v3, y_train_v3 = build_baseline_dataset_v3(train_indices)
X_test_v3, y_test_v3 = build_baseline_dataset_v3(test_fold)

xgb_model_v3 = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
xgb_model_v3.fit(X_train_v3, y_train_v3)
xgb_predictions_v3 = xgb_model_v3.predict(X_test_v3)

r_xgb_v3, _ = pearsonr(xgb_predictions_v3, y_test_v3)
is_ceiling_v3 = np.abs(y_test_v3 - 5.0) < 0.001
real_binder_r_xgb_v3, _ = pearsonr(xgb_predictions_v3[~is_ceiling_v3], y_test_v3[~is_ceiling_v3])
ci_xgb_v3 = concordance_index(y_test_v3, xgb_predictions_v3)

print(f'\nv3 (ITD-aware): R={r_xgb_v3:.4f} | Real-binder R={real_binder_r_xgb_v3:.4f} | CI={ci_xgb_v3:.4f}')
print(f'v2 (point-only): R={r_xgb_v2:.4f} | Real-binder R={real_binder_r_xgb_v2:.4f} | CI={ci_xgb_v2:.4f}')

Building v3 (ITD-aware) train/test sets...

v3 (ITD-aware): R=0.7124 | Real-binder R=0.5740 | CI=0.8378
v2 (point-only): R=0.7138 | Real-binder R=0.5788 | CI=0.8368


In [35]:
seeds_to_retry = [0, 1, 2, 3, 4]
lr_scheduled_results = []

for seed in seeds_to_retry:
    torch.manual_seed(seed)
    random.seed(seed)

    model = DTIModel(hidden_dim=64)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.3)
    loss_fn = nn.MSELoss()

    seed_train_indices = train_indices.copy()
    start = time.time()

    for epoch in range(8):
        random.shuffle(seed_train_indices)
        for flat_idx in seed_train_indices:
            drug_x, drug_edge_index, protein_indices, true_pKd, _, _ = get_example(flat_idx)
            optimizer.zero_grad()
            predicted_pKd = model(drug_x, drug_edge_index, protein_indices)
            loss = loss_fn(predicted_pKd, torch.tensor([true_pKd], dtype=torch.float))
            loss.backward()
            optimizer.step()
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        if epoch == 0 or (epoch+1) % 4 == 0:
            print(f'  seed {seed} epoch {epoch+1}: lr={current_lr:.6f}')

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for flat_idx in test_fold:
            drug_x, drug_edge_index, protein_indices, true_pKd, _, _ = get_example(flat_idx)
            preds.append(model(drug_x, drug_edge_index, protein_indices).item())
            trues.append(true_pKd)

    preds_arr, trues_arr = np.array(preds), np.array(trues)
    r, _ = pearsonr(preds_arr, trues_arr)
    ci = concordance_index(trues_arr, preds_arr, seed=seed)

    torch.save(model.state_dict(), f'../models/gnn_lrsched_seed{seed}.pt')
    lr_scheduled_results.append({'seed': seed, 'r': r, 'ci': ci})
    with open('../models/lr_scheduled_results.json', 'w') as f:
        json.dump(lr_scheduled_results, f)

    elapsed = (time.time()-start)/60
    print(f'Seed {seed}: R={r:.4f} | CI={ci:.4f} | time={elapsed:.1f}min\n')

rs = [x['r'] for x in lr_scheduled_results]
print(f'\nLR-scheduled mean R: {np.mean(rs):.4f} ± {np.std(rs):.4f}')

  seed 0 epoch 1: lr=0.001000
  seed 0 epoch 4: lr=0.000300
  seed 0 epoch 8: lr=0.000090
Seed 0: R=0.5425 | CI=0.7903 | time=24.6min

  seed 1 epoch 1: lr=0.001000
  seed 1 epoch 4: lr=0.000300
  seed 1 epoch 8: lr=0.000090
Seed 1: R=0.5559 | CI=0.7874 | time=22.7min

  seed 2 epoch 1: lr=0.001000
  seed 2 epoch 4: lr=0.000300
  seed 2 epoch 8: lr=0.000090
Seed 2: R=0.5258 | CI=0.7974 | time=22.8min

  seed 3 epoch 1: lr=0.001000
  seed 3 epoch 4: lr=0.000300
  seed 3 epoch 8: lr=0.000090
Seed 3: R=0.5642 | CI=0.7748 | time=22.6min

  seed 4 epoch 1: lr=0.001000
  seed 4 epoch 4: lr=0.000300
  seed 4 epoch 8: lr=0.000090
Seed 4: R=0.5432 | CI=0.7520 | time=26.0min


LR-scheduled mean R: 0.5463 ± 0.0131


In [3]:
import json
import numpy as np
import mlflow

In [4]:
with open('../models/lr_scheduled_results.json') as f:
    lr_scheduled_results = json.load(f)

rs = [x['r'] for x in lr_scheduled_results]
print('Loaded from disk:', lr_scheduled_results)
print(f'Mean R: {np.mean(rs):.4f} ± {np.std(rs):.4f}')

Loaded from disk: [{'seed': 0, 'r': 0.5425394483351508, 'ci': 0.7903066271018794}, {'seed': 1, 'r': 0.5559390381465183, 'ci': 0.7874015748031497}, {'seed': 2, 'r': 0.5257600047026895, 'ci': 0.797373358348968}, {'seed': 3, 'r': 0.56424182580598, 'ci': 0.7747572815533981}, {'seed': 4, 'r': 0.5431608373677079, 'ci': 0.752}]
Mean R: 0.5463 ± 0.0131


In [5]:
with mlflow.start_run(run_name="GNN_64dim_8epoch_LRscheduled_5seed"):
    mlflow.log_param("hidden_dim", 64)
    mlflow.log_param("num_epochs", 8)
    mlflow.log_param("lr_schedule", "StepLR_step4_gamma0.3")
    mlflow.log_param("model_type", "GNN")
    mlflow.log_metric("mean_pearson_r", np.mean(rs))
    mlflow.log_metric("std_pearson_r", np.std(rs))
    for seed, r_val in zip([0,1,2,3,4], rs):
        mlflow.log_metric(f"seed{seed}_r", r_val)

print("Final LR-scheduled result logged.")

Final LR-scheduled result logged.
